[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day15-fastapi-openai-compatible-server.ipynb)
# Day 15 — FastAPI OpenAI-Compatible Server
**Time:** ~60 min · **You need:** CPU or one T4 in Colab · **Prereqs:** Days 1–14

Today you build a real serving endpoint: an OpenAI-compatible `POST /v1/chat/completions`
server on FastAPI that streams tokens with correct SSE framing. Exit criteria:
1. `curl` the non-streaming endpoint and get a valid chat-completion JSON.
2. `curl -N` the streaming endpoint and see `data:` chunks ending in `data: [DONE]`.
3. Measure time-to-first-chunk (your TTFT) and 10-client p50/p99.
4. Measure JSON framing bytes per token.


In [ ]:
# Cell 1 — installs (run once)
!pip install -q fastapi "uvicorn[standard]" transformers torch requests
# Expected: installs complete, no errors. On a T4 runtime this pulls the CUDA torch build.


## Write the server
`server.py` below loads SmolLM2-135M once at import, exposes `/v1/chat/completions`
(streaming + non-streaming), `/v1/models`, and `/health`. Streaming uses
`TextIteratorStreamer` in a background thread so the async event loop keeps flushing chunks.


In [ ]:
# Cell 2 — write server.py to the notebook's working directory
from pathlib import Path
Path("server.py").write_text('import json, threading, time\nfrom fastapi import FastAPI\nfrom fastapi.responses import StreamingResponse\nfrom pydantic import BaseModel\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer\nimport torch\n\nMODEL_ID = "HuggingFaceTB/SmolLM2-135M"\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nprint("Device:", device)\n\napp = FastAPI()\ntok = AutoTokenizer.from_pretrained(MODEL_ID)\nmodel = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()\n\nclass ChatMessage(BaseModel):\n    role: str\n    content: str\n\nclass ChatCompletionRequest(BaseModel):\n    model: str = "smollm2-135m"\n    messages: list[ChatMessage]\n    max_tokens: int = 64\n    temperature: float = 0.7\n    stream: bool = False\n\ndef build_prompt(messages):\n    return "\\n".join(f"{m.role}: {m.content}" for m in messages) + "\\nassistant:"\n\n@app.get("/health")\ndef health():\n    return {"status": "ok"}\n\n@app.get("/v1/models")\ndef models():\n    return {"object": "list",\n            "data": [{"id": "smollm2-135m", "object": "model",\n                      "created": int(time.time())}]}\n\n@app.post("/v1/chat/completions")\nasync def chat_completions(req: ChatCompletionRequest):\n    prompt = build_prompt(req.messages)\n    inputs = tok(prompt, return_tensors="pt").to(device)\n    prompt_len = inputs.input_ids.shape[1]\n    gen_kwargs = dict(inputs, max_new_tokens=req.max_tokens)\n    if req.temperature and req.temperature > 0:\n        gen_kwargs["do_sample"] = True\n        gen_kwargs["temperature"] = req.temperature\n    else:\n        gen_kwargs["do_sample"] = False\n    if not req.stream:\n        out = model.generate(**gen_kwargs)\n        text = tok.decode(out[0][prompt_len:], skip_special_tokens=True)\n        return {"id": "chatcmpl-x", "object": "chat.completion",\n                "created": int(time.time()), "model": req.model,\n                "choices": [{"index": 0,\n                             "message": {"role": "assistant", "content": text},\n                             "finish_reason": "stop"}]}\n    streamer = TextIteratorStreamer(tok, skip_prompt=True,\n                                    skip_special_tokens=True)\n    gen_kwargs["streamer"] = streamer\n    thread = threading.Thread(target=model.generate,\n                              kwargs=gen_kwargs, daemon=True)\n    def event_gen():\n        thread.start()\n        for piece in streamer:\n            chunk = {"id": "chatcmpl-x", "object": "chat.completion.chunk",\n                     "created": int(time.time()), "model": req.model,\n                     "choices": [{"index": 0,\n                                  "delta": {"content": piece},\n                                  "finish_reason": None}]}\n            yield f"data: {json.dumps(chunk)}\\n\\n"\n        yield "data: [DONE]\\n\\n"\n    return StreamingResponse(event_gen(), media_type="text/event-stream")\n')
print("wrote", Path("server.py").resolve())
# Expected: wrote /content/server.py (or your cwd equivalent)


## Launch the server in the background
We start uvicorn as a subprocess and poll `/health` until it answers.
The server object lives on after this cell returns.


In [ ]:
# Cell 3 — launch uvicorn, wait for /health
import subprocess, time, requests, os
log = open("/tmp/uvicorn.log", "w")
proc = subprocess.Popen(["uvicorn", "server:app", "--port", "8000"],
                        stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd())
URL = "http://127.0.0.1:8000/v1/chat/completions"
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            print("server up"); break
    except Exception:
        pass
    time.sleep(2)
else:
    print("SERVER FAILED TO START — tail of log:")
    print(open("/tmp/uvicorn.log").read()[-2000:])
# Expected: 'server up' within ~30 s (model download on first run takes longer).


## Test 1 — non-streaming chat completion
One JSON object back, `choices[0].message.content` holds the full text.


In [ ]:
# Cell 4 — non-streaming request
import time, json
payload = {"model": "smollm2-135m",
           "messages": [{"role": "user", "content": "The capital of France is"}],
           "max_tokens": 10, "temperature": 0.0, "stream": False}
t0 = time.perf_counter()
r = requests.post(URL, json=payload, timeout=120)
e2e = time.perf_counter() - t0
r.raise_for_status()
body = r.json()
print("content:", body["choices"][0]["message"]["content"])
print(f"E2E latency for 10 tokens: {e2e*1000:.0f} ms")
print("top-level keys:", sorted(body.keys()))
# Expected: content starts with ' Paris'; E2E ~2-6 s on CPU, <1 s on T4.
# Record E2E here -> packet section 4.


## Test 2 — streaming (SSE)
`stream: true` returns `text/event-stream`: one `data: {...}` line per token,
each terminated by a blank line, ending with `data: [DONE]`. We time the
first chunk — that is the server's TTFT.


In [ ]:
# Cell 5 — streaming request, time to first chunk
import time
payload = {"model": "smollm2-135m",
           "messages": [{"role": "user", "content": "Count to five slowly"}],
           "max_tokens": 30, "temperature": 0.0, "stream": True}
t0 = time.perf_counter()
first_chunk_ms, chunks, done = None, [], False
with requests.post(URL, json=payload, stream=True, timeout=120) as r:
    r.raise_for_status()
    print("content-type:", r.headers["content-type"])
    for line in r.iter_lines():
        if not line:
            continue  # the blank-line SSE frame terminator
        if first_chunk_ms is None:
            first_chunk_ms = (time.perf_counter() - t0) * 1000
        if line == b"data: [DONE]":
            done = True; break
        chunks.append(line)
        if len(chunks) <= 3:
            print(line[:160])
print(f"chunks: {len(chunks)}, TTFT (time to first chunk): {first_chunk_ms:.0f} ms, DONE seen: {done}")
# Expected: content-type text/event-stream; TTFT ~= prefill + 1 decode step;
# ~30 chunks; DONE seen: True. Record TTFT here -> packet section 4.


## Test 3 — ten concurrent clients, no batching
The server handles one request at a time, so ten clients serialize.
Predict before running: p50 ≈ 5x solo, p99 ≈ 10x solo.


In [ ]:
# Cell 6 — concurrency hammer
import concurrent.futures, statistics, time
def one_client(i):
    t0 = time.perf_counter()
    r = requests.post(URL, json={"model": "smollm2-135m",
        "messages": [{"role": "user", "content": f"Say hello, client {i}"}],
        "max_tokens": 30, "temperature": 0.0, "stream": False}, timeout=300)
    r.raise_for_status()
    return time.perf_counter() - t0
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
    lats = sorted(ex.map(one_client, range(10)))
print("latencies (s):", [f"{x:.1f}" for x in lats])
print(f"p50: {statistics.median(lats):.1f} s, p99: {lats[-1]:.1f} s")
# Expected: p50 ~5x and p99 ~10x your Cell-4 solo latency — pure serialization.
# Record both here -> packet section 4. This is the number Day 17's batching beats.


## Test 4 — SSE framing overhead
Count the JSON framing bytes per streamed token. Predict ~100-200 B/token.


In [ ]:
# Cell 7 — framing bytes per token
framing_bytes, ntok = 0, 0
with requests.post(URL, json={"model": "smollm2-135m",
    "messages": [{"role": "user", "content": "Name three planets"}],
    "max_tokens": 30, "temperature": 0.0, "stream": True}, stream=True,
    timeout=120) as r:
    for line in r.iter_lines():
        if line.startswith(b"data: "):
            framing_bytes += len(line)
            if line != b"data: [DONE]":
                ntok += 1
print(f"{framing_bytes} framing bytes / {ntok} tokens = {framing_bytes/max(ntok,1):.0f} B/token")
# Expected: ~100-200 bytes of JSON framing per token. Record -> packet section 4.


## Wrap-up — record your numbers
Fill the packet's §4 table with: solo TTFT (Cell 5), solo E2E (Cell 4),
10-client p50/p99 (Cell 6), framing bytes/token (Cell 7), solo vs aggregate tok/s.

**Checkpoints** (answers in the Day 15 packet):
1. What exact bytes terminate one SSE chunk, and what ends the stream?
2. 200-token response, TTFT 55 ms, TPOT 5 ms: perceived latency with vs without streaming?
3. ~150 B framing per 6-B token — fine at the edge, but why not internally?
4. 10 concurrent clients, 1 s service each, no batching: p99?
5. Why must `generate()` run in a thread when streaming from an async route?

**Cleanup (optional):** `proc.terminate()` stops the background server.
Tomorrow (Day 16): admission queue with backpressure (429 when full) and
per-stage latency logging — queue wait, prefill, decode.
